# AutoJev-style Qwen3.8-27B reproduction

This notebook reproduces the completed 27B text-only experiment with the public AutoJev data builder, the repository's deterministic gap curriculum, a 255-way learned decision readout, option-order augmentation, LoRA, and held-out temperature scaling.

**Hardware:** the reference run used an NVIDIA RTX PRO 6000 Blackwell (98 GB). Peak allocated GPU memory was 52.43 GiB. Use a GPU with at least 64 GB usable memory for microbatch 4, or start with microbatch 1. Colab accelerator availability varies.

Checkpoints are written directly to Google Drive. No API key is required for training. `TYPESAFE_API_KEY` is only required for the optional live Jev comparison.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))


## Mount Drive and fetch pinned source

Authorize Drive in this foreground cell. The public repository and AutoJev are checked out at immutable revisions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/qwen-rlcd-jev /content/autojev
!git clone https://github.com/jsaurabh/qwen-rlcd-jev-repro.git /content/qwen-rlcd-jev
!git clone https://github.com/denis-pplx/autojev.git /content/autojev
!git -C /content/autojev checkout --detach ee63c1515980491a742f0bd0685c8dc5ca1f00c3
!python -m pip uninstall -y torchaudio torchao
!python -m pip install -q -e /content/autojev 'peft>=0.17'

# AutoJev 0.2.0 assumes every generated split is nonempty. This text-only run
# intentionally requests zero images, so make max(empty) well-defined.
from pathlib import Path
p=Path('/content/autojev/src/autojev/data.py')
s=p.read_text().replace(
    '"maximum_input_tokens": max(cast(int, row["source"]["input_tokens"]) for row in rows),',
    '"maximum_input_tokens": max((cast(int, row["source"]["input_tokens"]) for row in rows), default=0),'
)
p.write_text(s)


## Build and freeze the training/calibration splits

The upstream half-scale text corpus contributes about 47,500 training rows. The committed gap curriculum adds 1,920 rule-labeled counterfactual rows. The filter keeps only complete branches at or below 512 tokens; it never truncates. The reference run retained 49,039 of 49,420 mixed training rows, 1,440 of 1,497 development rows, and 945 of 983 calibration rows.


In [ ]:
!rm -rf /content/autojev-data
!autojev-data --output /content/autojev-data --manifest /content/autojev-data-manifest.json --train-scale 0.5 --image-train 0 --image-eval 0 --max-length 8192
!python /content/qwen-rlcd-jev/colab/build_autojev_mix.py --autojev-data /content/autojev-data --gap-data /content/qwen-rlcd-jev/data/jev_gap_curriculum_v1 --max-length 512
!cat /content/autojev-data/filter-report.json


## One-step memory and code-path preflight

This validates model loading, gradients, artifact saving, and calibration before spending hours on the full run.


In [ ]:
!python -u /content/qwen-rlcd-jev/colab/autojev_lora_train.py \
  --data /content/autojev-data \
  --train-file train-mixed.jsonl \
  --dev-file dev-filtered.jsonl \
  --temperature-file temperature-filtered.jsonl \
  --steps 1 --micro-batch 1 --grad-accum 1 --eval-rows 4 --save-every 1 \
  --out /content/drive/MyDrive/qwen-rlcd-jev/autojev-27b/repro-preflight


## Full run

The exact reference configuration is 1,533 optimizer steps, microbatch 4, accumulation 8, rank 16, and learning rate `5e-5`. On the reference GPU the completed resumed segment took 11,786 seconds (3.27 hours). Leave the browser connected and inspect `nvidia-smi` in a Colab terminal if desired.


In [ ]:
!python -u /content/qwen-rlcd-jev/colab/autojev_lora_train.py \
  --data /content/autojev-data \
  --train-file train-mixed.jsonl \
  --dev-file dev-filtered.jsonl \
  --temperature-file temperature-filtered.jsonl \
  --steps 1533 --micro-batch 4 --grad-accum 8 --eval-rows 512 --save-every 100 \
  --out /content/drive/MyDrive/qwen-rlcd-jev/autojev-27b/reproduction-v1


## Resume after interruption

Set `checkpoint` to the latest complete Drive checkpoint. New public checkpoints contain `training_state.pt`, so AdamW moments and RNG state resume. The historical run resumed from step 300 without optimizer state; its frozen report records that limitation.


In [ ]:
from pathlib import Path
import subprocess

checkpoint = Path('/content/drive/MyDrive/qwen-rlcd-jev/autojev-27b/reproduction-v1/checkpoint-00300')
start_step = int(checkpoint.name.split('-')[-1])
command = [
    '/usr/bin/python3', '-u', '/content/qwen-rlcd-jev/colab/autojev_lora_train.py',
    '--data', '/content/autojev-data',
    '--train-file', 'train-mixed.jsonl',
    '--dev-file', 'dev-filtered.jsonl',
    '--temperature-file', 'temperature-filtered.jsonl',
    '--steps', '1533', '--start-step', str(start_step),
    '--resume-artifact', str(checkpoint),
    '--micro-batch', '4', '--grad-accum', '8', '--eval-rows', '512', '--save-every', '100',
    '--out', '/content/drive/MyDrive/qwen-rlcd-jev/autojev-27b/reproduction-v1',
]
print(' '.join(command))
# subprocess.run(command, check=True)  # uncomment only to resume


## Optional comparison against Jev

Add `TYPESAFE_API_KEY` in Colab Secrets and enable notebook access. Read it in the notebook process, then pass it through the subprocess environment. The evaluator caches Jev responses and evaluates the local model and Jev on identical rows.


In [ ]:
import os, subprocess
from google.colab import userdata

env = os.environ.copy()
env['TYPESAFE_API_KEY'] = userdata.get('TYPESAFE_API_KEY')
subprocess.run([
    '/usr/bin/python3', '-u', '/content/qwen-rlcd-jev/colab/eval_autojev27b_vs_jev.py',
    '--artifact', '/content/drive/MyDrive/qwen-rlcd-jev/autojev-27b/reproduction-v1/selected',
    '--eval-file', '/content/qwen-rlcd-jev/data/jev_gap_curriculum_v1/eval.jsonl',
    '--output', '/content/drive/MyDrive/qwen-rlcd-jev/autojev-27b/reproduction-v1/eval',
    '--limit', '128', '--batch-size', '4', '--workers', '4', '--jev-model', 'jev-1.13.0',
], check=True, env=env)
